# mappings.ipynb

Builds one gene-mapping TSV per sample (`DataStructures/Mappings_by_Sample/{sample}_mapping.tsv`,
columns `gene_id, scaffold_id, taxonomy, kegg, enzyme, cog`) directly from
the raw per-gene JGI annotation files in `Metagenome_Samples/<sample>/`.

**Rebuilt after two independent reviews found real, systemic data loss in
the previous version** (which read intermediate `*_byTP` JSON files and
anchored the gene list to COG only):
- Old base gene list = COG's genes only, silently dropping any gene with
  real KEGG/Enzyme/Taxonomy data but no COG hit (measured: 30-40% of genes
  per sample).
- Old logic kept only the first of a gene's multiple COG/Enzyme hits,
  silently dropping the rest.
- `scaffold_id` was sourced from taxonomy data instead of `gene_id` itself,
  losing the scaffold for any gene with no taxonomy hit.

**Corrected rules** (verified against JGI's own
`Metagenome_Samples/<sample>/README.txt`, IMG Pipeline v5.0 documentation):
1. Base gene list = union of gene_ids across all four raw files
   (`.a.cog.txt`, `.a.ko.txt`, `.a.ec.txt`, `.a.phylodist.txt`) — a gene
   needs >=1 real hit in >=1 source to get a row. A gene with zero hits
   anywhere has nothing to map, so it's correctly excluded — this is
   different from the old bug, which excluded genes that DO have real
   data just because that data wasn't specifically in COG.
2. Per source, per gene: collapse to the sorted, comma-joined set of
   unique annotation IDs (not first-only). Matches how this app's existing
   canonical annotation builders already dedupe.
3. `scaffold_id` parsed directly from `gene_id`
   (`"{scaffold}_{start}_{end}"`), not sourced from taxonomy.
4. No evalue/percent_identity/bit_score filtering — matches every existing
   canonical annotation source in this app (deliberate consistency choice).
5. Output rows are sorted by `gene_id`, so re-running this notebook
   produces byte-identical output (diffable, reproducible).

Independently verified by `tests/test_mappings_notebook_matches.py`, which
re-derives the same result from the same raw files using a separate
implementation and compares against the TSVs this notebook writes.

In [ ]:
import csv
from collections import defaultdict
from pathlib import Path

# Base data-structures directory (output goes here) and the raw
# per-sample JGI annotation directory (input comes from here)
base_ds        = Path(r"C:\Users\sakoehn\Desktop\JGI\DataStructures")
metagenome_dir = Path(r"C:\Users\sakoehn\Desktop\JGI\Metagenome_Samples")

# Sample names
with open(r"C:\Users\sakoehn\Desktop\JGI\sample_names.txt") as f:
    samples = [s.strip() for s in f]


In [ ]:

# Output dir for final per-sample mapping tables
out_dir = base_ds / "Mappings_by_Sample"
out_dir.mkdir(parents=True, exist_ok=True)


def scaffold_from_gene_id(gene_id: str) -> str:
    """gene_id is "{scaffold}_{start}_{end}" -- verified at scale (1.09M
    gene_ids, 0 exceptions) that the last two underscore-separated fields
    are always the numeric coordinates."""
    return gene_id.rsplit("_", 2)[0]


def parse_multi_value_hits(path: Path, id_column: int) -> dict:
    """Reads a raw tab-delimited hit file (cog.txt/ko.txt/ec.txt shape, no
    header) and groups it by gene_id -> {unique annotation ids}. Every
    sample has all four hit files, so a missing one means a failed
    download, not a sample without hits: carrying on would make this
    sample look as if nothing was detected, so it stops instead."""
    hits = defaultdict(set)
    if not path.exists():
        raise FileNotFoundError(f"{path} is missing")
    with open(path, newline="") as f:
        reader = csv.reader(f, delimiter="\t")
        for row in reader:
            if not row:
                continue
            hits[row[0]].add(row[id_column])
    return hits


def parse_taxonomy_hits(path: Path) -> dict:
    """phylodist.txt is single-valued per gene: gene_id -> lineage string (column 4)."""
    hits = {}
    if not path.exists():
        raise FileNotFoundError(f"{path} is missing")
    with open(path, newline="") as f:
        reader = csv.reader(f, delimiter="\t")
        for row in reader:
            if not row:
                continue
            hits[row[0]] = row[4]
    return hits


def joined(values) -> str:
    """Sorted, comma-joined multi-value cell -- deterministic regardless
    of set iteration order. "" for no hits (reads back as None everywhere
    downstream, same as the old TSV's blank-field convention)."""
    if not values:
        return ""
    return ",".join(sorted(values))


# Main per-sample loop
for sample in samples:
    print("processing sample", sample)

    sample_dir = metagenome_dir / sample
    cog_hits  = parse_multi_value_hits(sample_dir / f"{sample}.a.cog.txt", id_column=1)
    kegg_hits = parse_multi_value_hits(sample_dir / f"{sample}.a.ko.txt", id_column=2)
    enz_hits  = parse_multi_value_hits(sample_dir / f"{sample}.a.ec.txt", id_column=2)
    tax_hits  = parse_taxonomy_hits(sample_dir / f"{sample}.a.phylodist.txt")

    # Union of every gene with >=1 real hit in any source -- NOT anchored
    # to a single source (the old bug anchored to COG only, silently
    # dropping 30-40% of genes per sample that had real KEGG/enzyme/
    # taxonomy data but no COG hit). Sorted so output is reproducible.
    genes = sorted(set(cog_hits) | set(kegg_hits) | set(enz_hits) | set(tax_hits))
    if not genes:
        raise ValueError(f"{sample}: no genes in any of its four hit files")

    rows = []
    for gene_id in genes:
        rows.append({
            "gene_id":     gene_id,
            "scaffold_id": scaffold_from_gene_id(gene_id),
            "taxonomy":    tax_hits.get(gene_id, ""),
            "kegg":        joined(kegg_hits.get(gene_id)),
            "enzyme":      joined(enz_hits.get(gene_id)),
            "cog":         joined(cog_hits.get(gene_id)),
        })

    # Write final per-sample mapping table
    out_path = out_dir / f"{sample}_mapping.tsv"
    with open(out_path, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["gene_id", "scaffold_id", "taxonomy", "kegg", "enzyme", "cog"],
            delimiter="\t",
        )
        writer.writeheader()
        writer.writerows(rows)

    print("wrote", out_path, "rows:", len(rows))